In [1]:
import sys

from pathlib import Path

import pickle
import pandas as pd
import numpy as np

sys.path.append("../")

from rec_tools.ranking_metrics import (
    map_at_k,
    hit_ratio_at_k,
    binary_ndcg_at_k,
    shuffle_arrays_consistently,
)

In [2]:
RESULTS_DIR = Path("../results")

In [3]:
binary_results_dir = RESULTS_DIR / "binary_results"
regression_results_dir = RESULTS_DIR / "regression_results"

binary_results_df = pd.read_csv(binary_results_dir / "binary_metrics.csv")
regression_results_df = pd.read_csv(regression_results_dir / "regression_metrics.csv")

In [4]:
binary_styled = binary_results_df.style.highlight_min(
    color="lightgreen", subset=["val_loss"]
).highlight_max(color="lightgreen", subset=["acc", "f1"])

binary_styled

,experiment,acc,f1,val_loss
0,results_svd_with_default_params_ts_binary,0.693174,0.749206,0.638989
1,results_lgb_with_default_params_li_binary,0.701656,0.734922,0.573969
2,results_ctb_with_native_feature_selection_ch_ts_shap_binary,0.681989,0.746544,0.597840
3,results_lgb_with_optuna_ch_ts_binary,0.683659,0.723104,0.595649
4,results_ctb_with_native_feature_selection_ch_li_shap_binary,0.703477,0.737966,0.562538
5,results_ctb_with_hyperopt_ch_ts_binary,0.675079,0.706349,0.601821
6,results_lgb_with_ray_ch_li_binary,0.712086,0.744227,0.561117
7,results_ctb_with_default_params_li_binary,0.696689,0.736782,0.570283
8,results_ctb_with_text_and_default_params_ts_binary,0.689653,0.751763,0.587906
9,results_lgb_with_feature_elimination_ch_li_binary,0.704801,0.739060,0.564565


In [5]:
regression_styled = regression_results_df.style.highlight_min(
    color="lightgreen", subset=["rmse"]
)

regression_styled

,experiment,rmse,val_loss
0,results_ctb_with_ray_ch_ts_regression,1.013439,1.013439
1,results_ctb_with_feature_elimination_ch_li_regression,0.884137,0.884137
2,results_lgb_with_optuna_ch_ts_regression,0.972196,0.972196
3,results_svd_with_hyperopt_li_regression,0.845652,0.845652
4,results_ctb_with_default_params_ts_regression,0.950983,0.950983
5,results_lgb_with_optuna_ch_li_regression,0.888253,0.888253
6,results_svd_with_hyperopt_ts_regression,0.929571,0.929571
7,results_ctb_with_default_params_li_regression,0.909076,0.909076
8,results_ctb_with_native_feature_selection_ch_ts_shap_regression,0.989013,0.989013
9,results_ctb_with_ray_ch_li_regression,0.891346,0.891346


In [6]:
def load_ranking_results(base_dir: Path) -> dict:

    results = {}

    # Find all directories containing 'ranking_metrics'
    ranking_dirs = [
        d for d in base_dir.glob("*") if d.is_dir() and "ranking_metrics" in d.name
    ]

    # Process each ranking metrics directory
    for ranking_dir in ranking_dirs:
        # Find all results.pkl files in subdirectories
        for pkl_path in ranking_dir.glob("*/results.pkl"):
            # Get experiment name (parent directory name)
            experiment_name = pkl_path.parent.name

            # Load pickle file
            with open(pkl_path, "rb") as f:
                results[experiment_name] = pickle.load(f)

    return results

In [7]:
ranking_results = load_ranking_results(RESULTS_DIR)

In [8]:
def create_metrics_df(results_dict: dict) -> pd.DataFrame:
    """
    Convert nested results dictionary into a DataFrame with metrics at different N values.
    """
    rows = []

    for experiment_name, metrics in results_dict.items():
        row = {"experiment_name": experiment_name}

        # Add metrics for each N
        for n in [5, 10, 20]:
            if n in metrics:
                row[f"ndcg@{n}"] = metrics[n]["ndcg"]
                row[f"map@{n}"] = metrics[n]["map"]
                row[f"hr@{n}"] = metrics[n]["hr"]

        rows.append(row)

    # Create DataFrame with specific column order
    columns = ["experiment_name"] + [
        f"{metric}@{n}" for n in [5, 10, 20] for metric in ["ndcg", "map", "hr"]
    ]

    return pd.DataFrame(rows).reindex(columns=columns)


# Create the DataFrame
metrics_df = create_metrics_df(ranking_results)

In [9]:
metrics_styled = metrics_df.style.highlight_max(
    color="lightgreen",
    subset=[
        "ndcg@5",
        "map@5",
        "hr@5",
        "ndcg@10",
        "map@10",
        "hr@10",
        "ndcg@20",
        "map@20",
        "hr@20",
    ],
)
metrics_styled

,experiment_name,ndcg@5,map@5,hr@5,ndcg@10,map@10,hr@10,ndcg@20,map@20,hr@20
0,results_lgb_with_default_params_li_binary,0.147737,0.120086,0.232624,0.193868,0.139004,0.375956,0.245663,0.153128,0.581144
1,results_lgb_with_ray_ch_li_binary,0.138215,0.114766,0.210176,0.178667,0.131376,0.335717,0.229441,0.145086,0.538244
2,results_lgb_with_feature_elimination_ch_ts_regression,0.123433,0.095552,0.209345,0.169732,0.114659,0.352511,0.216005,0.127130,0.537246
3,results_lgb_with_hyperopt_ch_li_regression,0.176438,0.147631,0.264549,0.219074,0.165098,0.397073,0.263063,0.177050,0.571999
4,results_lgb_with_feature_elimination_ch_ts_binary,0.122267,0.094843,0.207017,0.167936,0.113608,0.348188,0.215248,0.126412,0.536581
5,results_lgb_with_default_params_li_regression,0.149969,0.122486,0.234120,0.197314,0.141946,0.381111,0.248210,0.155847,0.583638
6,results_lgb_with_default_params_ts_regression,0.153977,0.125848,0.240273,0.202732,0.145837,0.391919,0.250892,0.158960,0.582973
7,results_lgb_with_default_params_ts_binary,0.157168,0.129115,0.243432,0.205320,0.148893,0.392584,0.253333,0.161969,0.583638
8,results_svd_with_default_params_ts_binary,0.113007,0.087025,0.189890,0.159895,0.106205,0.339042,0.208133,0.124739,0.532092
9,results_svd_with_hyperopt_li_regression,0.216140,0.182072,0.319421,0.263101,0.201439,0.464915,0.310050,0.214397,0.650648


In [18]:
best_y_pred = np.array(
    [
        [0.8, 0.6, 0.4, 0.3, 0.1],
        [0.7, 0.6, 0.5, 0.2, 0.1],
    ]
)

worst_y_pred = np.array(
    [
        [0.1, 0.2, 0.4, 0.3, 0.8],
        [0.1, 0.2, 0.3, 0.4, 0.5],
    ]
)

y_true = np.array(
    [
        [1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0],
    ]
)

k = 5

In [19]:
round(
    binary_ndcg_at_k(best_y_pred.reshape(-1, 1), y_true.reshape(-1, 1), n_items=5, k=5),
    2,
)

0.82

In [12]:
round(
    binary_ndcg_at_k(
        worst_y_pred.reshape(-1, 1), y_true.reshape(-1, 1), n_items=5, k=5
    ),
    2,
)

0.39

In [13]:
discounts = 1.0 / np.log2(np.arange(2, k + 2))

discounts

array([1.        , 0.63092975, 0.5       , 0.43067656, 0.38685281])

In [14]:
map_at_k(best_y_pred.reshape(-1, 1), y_true.reshape(-1, 1), n_items=5, k=5)

1.0

In [15]:
# the precision at 5 is 0.2
map_at_k(worst_y_pred.reshape(-1, 1), y_true.reshape(-1, 1), n_items=5, k=5)

0.2